# Positional Encoding 심층 분석 - 실습 코드 2: Sinusoidal vs Learned vs RoPE 위치 인코딩 비교 실험

- Tutorial ID: `expand-positional-encoding`
- Tutorial: Positional Encoding 심층 분석
- Section ID: `expand-positional-encoding-code-2`
- Section: 실습 코드 2: Sinusoidal vs Learned vs RoPE 위치 인코딩 비교 실험


In [ ]:
# ============================================================
# 코드 읽는 법 — 실습 코드 2: Sinusoidal vs Learned vs RoPE 위치 인코딩 비교 실험
#
# 이 코드는 “정답을 한 번 실행”하는 용도가 아니라,
# 수학/아키텍처 개념이 실제 배열·텐서 연산으로 바뀌는 과정을
# 한 줄씩 추적하기 위한 실험 노트입니다.
#
# 학습 목표:
#   1) Q/K/V가 어떤 shape으로 만들어지고 attention score로 이어지는지 추적
#   2) 위치 정보가 벡터 회전/편향으로 attention score에 들어가는 방식 확인
#
# 읽는 순서:
#   1) 차원/하이퍼파라미터(batch_size, seq_len, d_model 등)를 먼저 확인합니다.
#   2) 입력 배열 X 또는 토큰/문서 데이터가 어떻게 만들어지는지 봅니다.
#   3) W_Q/W_K/W_V/W_O 같은 가중치 행렬이 어떤 공간으로 투영하는지 확인합니다.
#   4) @, matmul, softmax, mask, loss 등 핵심 연산 직후의 shape와 값을 출력으로 검증합니다.
#   5) seed, 차원, temperature, rank, top_k, expert 수 등을 바꿔 결과가 어떻게 변하는지 실험합니다.
#
# 주의:
#   - 숫자 하나하나를 외우기보다 “shape 변화”와 “정보가 이동하는 방향”을 보세요.
#   - torch/transformers/openai/vLLM 의존 코드는 Colab/로컬/서버 노트북 실행을 권장합니다.

In [ ]:
import torch
import torch.nn as nn
import math
import numpy as np

# ── 1. Sinusoidal Positional Encoding ──
class SinusoidalPE(nn.Module):
    """원래 Transformer의 위치 인코딩"""
        def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

        def forward(self, x):
        return x + self.pe[:, :x.size(1)]

# ── 2. Learned Positional Embedding ──
class LearnedPE(nn.Module):
    """BERT/GPT-2 스타일 학습 가능 위치 임베딩"""
        def __init__(self, d_model, max_len=512):
        super().__init__()
        self.embedding = nn.Embedding(max_len, d_model)
    
        def forward(self, x):
                seq_len = x.size(1)
        positions = torch.arange(seq_len, device=x.device)
        return x + self.embedding(positions)

# ── 3. ALiBI Positional Bias ──
class ALiBiAttention(nn.Module):
    """ALiBI: Attention with Linear Biases"""
        def __init__(self, n_heads, d_model):
        super().__init__()
        # 헤드별 스lopes (기하급수적으로 감소)
        # m_h = 2^(-8/n * h) for h = 1, ..., n
        slopes = 2 ** (-8 / n_heads * torch.arange(1, n_heads + 1))
        self.register_buffer('slopes', slopes)
        self.n_heads = n_heads
        self.d_model = d_model
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
    
        def forward(self, x):
        B, N, D = x.shape
        d_k = D // self.n_heads
        
                Q = self.q_proj(x).view(B, N, self.n_heads, d_k).transpose(1, 2)
                K = self.k_proj(x).view(B, N, self.n_heads, d_k).transpose(1, 2)
                V = self.v_proj(x).view(B, N, self.n_heads, d_k).transpose(1, 2)
        
        # Standard attention scores
                scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
        
        # ALiBI: 위치 편향 추가
        # bias[i, j] = m_h * -(i - j)  for i >= j (causal)
        positions = torch.arange(N, device=x.device)
        distance = positions.unsqueeze(0) - positions.unsqueeze(1)  # (N, N)
        # 인과적 마스크 + 편향
                causal_mask = (distance >= 0).float()
        alibi_bias = self.slopes.view(-1, 1, 1) * distance.unsqueeze(0) * (-1)
        alibi_bias = alibi_bias * causal_mask.unsqueeze(0) + (1 - causal_mask.unsqueeze(0)) * (-1e9)
        
                scores = scores + alibi_bias
        attn = torch.softmax(scores, dim=-1)
                output = torch.matmul(attn, V)
        return output.transpose(1, 2).contiguous().view(B, N, D)

# ── 비교 실험: 외삽 능력 테스트 ──
print("=== 위치 인코딩 비교 실험 ===\n")

d_model = 128
n_heads = 4
train_len = 64   # 학습 시 시퀀스 길이
eval_len = 128   # 평가 시 시퀀스 길이 (2배 외삽)

# 더미 입력
x_train = torch.randn(2, train_len, d_model)
x_eval = torch.randn(2, eval_len, d_model)

# 1. Sinusoidal
sin_pe = SinusoidalPE(d_model, max_len=5000)
sin_out_train = sin_pe(x_train)
sin_out_eval = sin_pe(x_eval)  # 항상 가능 (고정 공식)
print(f"Sinusoidal: 학습 {sin_out_train.shape} → 외삽 {sin_out_eval.shape} ✅")

# 2. Learned (max_len=100)
learned_pe = LearnedPE(d_model, max_len=100)
learned_out_train = learned_pe(x_train)
try:
    learned_out_eval = learned_pe(x_eval)  # max_len 초과 시 에러
    print(f"Learned:    학습 {learned_out_train.shape} → 외삽 {learned_out_eval.shape} ✅")
except Exception as e:
    print(f"Learned:    학습 {learned_out_train.shape} → 외삽 ❌ (max_len 초과)")

# 3. ALiBI (길이 무관)
alibi = ALiBiAttention(n_heads, d_model)
alibi_out_train = alibi(x_train)
alibi_out_eval = alibi(x_eval)  # 길이 무관
print(f"ALiBI:      학습 {alibi_out_train.shape} → 외삽 {alibi_out_eval.shape} ✅")

print("\n=== 요약 ===")
print("Sinusoidal: 외삽 가능하지만 성능 저하")
print("Learned:    외삽 불가 (max_len 고정)")
print("RoPE:       외삽 가능 (NTK-aware scaling)")
print("ALiBI:      외삽 가능 (길이 무관, 추가 학습 불필요)")